# Create a normalized dataset for 20 radiologist-labelled studies

The 20-study radiologist labels from `excluded_labels.csv` are embedded below. **Only attach `gany24558/rsna-knee-normalized-all-training-data`** as an input dataset. CPU is sufficient; no DICOM decoding, normalization or GPU is needed. Enable Internet for the final publishing cell.

The embedded snapshot contains 20 unique study IDs and all 12 binary targets. Selection comes from these labels, not the 30 folders in the local verified-studies directory. The snapshot is independent of future edits to the Desktop CSV. To update it, replace the embedded text and its SHA-256 together after reviewing the new labels. Keep this notebook and its output private.

This notebook creates a standalone output folder with only the selected images and verified-label tables. The final cell publishes it to `gany24558/rsna-knee-radiologist-verified-20`. It does not change the source dataset. No train/validation split is assigned; keep these studies out of model training if using them for independent evaluation.


In [1]:
from pathlib import Path
import json, hashlib, io, os, shutil, tarfile, tempfile
from contextlib import ExitStack
import numpy as np
import pandas as pd

NORMALIZED_ROOT = Path('/kaggle/input/datasets/gany24558/rsna-knee-normalized-all-training-data')
OUTPUT_DIR = Path('/kaggle/working/rsna-knee-radiologist-20')
EXPECTED_STUDIES = 20
MIN_FREE_GB = 1.0
SOURCE_HANDLE = 'gany24558/rsna-knee-normalized-all-training-data'
ID, SID = 'StudyInstanceUID', 'SeriesInstanceUID'
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
           'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


In [2]:
# Verified snapshot of excluded_labels.csv; 20 study rows, 12 targets.
RADIOLOGIST_LABELS_TEXT = '''StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866,0,0,0,0,0,0,0,0,0,0,0,0
1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097,1,0,0,0,0,0,0,1,0,0,1,0
1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636,0,0,0,0,0,0,0,0,0,0,0,0
1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471,0,0,0,0,0,0,0,0,0,0,0,0
1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773,0,0,0,0,0,0,0,0,0,0,0,0
1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801,1,0,1,0,0,0,0,1,0,1,0,0
1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585,0,0,0,0,0,0,0,0,0,0,0,0
1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062,0,0,0,0,0,0,0,0,0,0,0,0
1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576,0,0,0,0,0,0,0,1,0,0,1,0
1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029,0,0,0,0,0,0,1,1,1,0,0,0
1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306,0,0,0,0,0,0,0,0,0,0,0,0
1.2.826.0.1.3680043.8.498.10352044313309892076985642485759368197,0,0,0,1,0,0,0,1,0,0,1,0
1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066,1,0,0,0,0,0,0,1,0,0,1,0
1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933,0,0,0,0,1,0,1,0,1,1,0,0
1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674,0,0,0,0,0,0,1,1,1,0,0,0
1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522,1,0,0,0,0,0,1,1,0,1,1,0
1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946,0,1,1,1,1,0,1,1,0,1,1,1
1.2.826.0.1.3680043.8.498.10524164196949381411165626176732559196,0,0,1,1,1,1,0,1,1,1,0,0
1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828,0,0,0,0,0,0,1,1,0,0,0,0
1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573,0,0,0,0,0,0,0,0,0,0,0,0'''
EMBEDDED_LABELS_SHA256 = '8d9123a1bcb0a1e9a01372b81397aebfb7955e3d232492d897f4066eadf9c238'


## 1. Read and validate the completed labels
Blank targets stay unknown. Known radiologist targets receive mask=1 and weight=1. Source dataset labels are never used as replacement targets.

In [3]:
def read_verified_labels(csv_text, expected=20):
    frame = pd.read_csv(io.StringIO(csv_text), dtype={ID: str})
    missing = set([ID, *TARGETS]) - set(frame.columns)
    if missing:
        raise ValueError(f'Missing label columns: {sorted(missing)}')
    frame = frame[[ID, *TARGETS]].copy()
    frame[ID] = frame[ID].str.strip()
    if frame[ID].isna().any() or frame[ID].eq('').any() or not frame[ID].is_unique:
        raise ValueError('Study IDs must be nonempty and unique.')
    if len(frame) != expected:
        raise ValueError(f'Expected {expected} completed studies, found {len(frame)}. Filter the label export explicitly.')
    for target in TARGETS:
        frame[target] = pd.to_numeric(frame[target], errors='raise')
        known = frame[target].notna()
        if not frame.loc[known, target].isin([0, 1]).all():
            raise ValueError(f'{target}: use only 0, 1 or blank; uncertain labels must not be forced negative.')
    if not frame[TARGETS].notna().any(axis=1).all():
        raise ValueError('Every selected study must have at least one known radiologist target.')
    return frame.sort_values(ID).reset_index(drop=True)

assert hashlib.sha256(RADIOLOGIST_LABELS_TEXT.encode('utf-8')).hexdigest() == EMBEDDED_LABELS_SHA256, 'Embedded label snapshot changed; review labels and update its checksum.'
verified = read_verified_labels(RADIOLOGIST_LABELS_TEXT, EXPECTED_STUDIES)
print('Selected radiologist-labelled studies:', len(verified))
display(verified[TARGETS].agg(['count', 'sum']).rename(index={'count':'known targets', 'sum':'positive targets'}))


Selected radiologist-labelled studies: 20


,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
known targets,20,20,20,20,20,20,20,20,20,20,20,20
positive targets,4,1,3,3,3,1,6,11,4,5,6,1


## 2. Inspect the active source shards
Use full series manifests, including studies excluded from the original supervised selection. Reject missing, failed or pending selected series rather than silently making a smaller dataset. Legacy archives are ignored.

In [4]:
def safe_file(archive, name):
    rel = Path(name)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError(f'Unsafe archive member: {name}')
    member = archive.getmember(name)
    if not member.isfile():
        raise ValueError(f'Expected a regular file: {name}')
    return member

def inspect_source(root, selected_ids):
    root = Path(root)
    index = json.loads((root / 'dataset_index.json').read_text())
    if index.get('dataset_handle') != SOURCE_HANDLE:
        raise ValueError('Attach the all-training-data dataset, not the earlier supervised-only dataset.')
    identity = index['identity']
    expected = {f'{i:03d}' for i in range(int(identity['num_shards']))}
    if set(index['shards']) != expected:
        raise ValueError('Source dataset is missing active shards.')
    frames, config = [], None
    for slot, entry in sorted(index['shards'].items()):
        if entry['status'] != 'processed' or Path(entry['file']).name != entry['file']:
            raise ValueError(f'Invalid or incomplete source shard {slot}')
        package = root / entry['file']
        with tarfile.open(package, 'r:') as arc:
            info = json.load(arc.extractfile(safe_file(arc, 'publication_manifest.json')))
            if info['status'] != 'processed' or any(info.get(k) != v for k, v in identity.items()):
                raise ValueError(f'Shard {slot}: publication identity mismatch')
            prep = json.load(arc.extractfile(safe_file(arc, 'preprocessing_config.json')))
            if prep['run_id'] != identity['run_id'] or (config is not None and prep != config):
                raise ValueError('Source preprocessing configurations differ.')
            config = prep
            name = f'series_manifest_shard_{int(slot):03d}.csv'
            frame = pd.read_csv(arc.extractfile(safe_file(arc, name)), dtype={ID: str, SID: str})
            frame = frame[frame[ID].isin(selected_ids)].copy()
            frame['source_package'] = entry['file']
            for row in frame.loc[frame.status.eq('ok')].itertuples():
                safe_file(arc, row.cache_path)
            frames.append(frame)
        print('Inspected shard', slot, '- selected series:', len(frame))
    selected = pd.concat(frames, ignore_index=True)
    if selected.duplicated([ID, SID]).any():
        raise ValueError('Duplicate selected study/series IDs across shards.')
    missing = set(selected_ids) - set(selected[ID])
    if missing:
        raise ValueError(f'Selected studies missing from source manifests: {sorted(missing)}')
    failed = selected.loc[~selected.status.eq('ok'), [ID, SID, 'status', 'error']]
    if len(failed):
        display(failed)
        raise ValueError('Selected studies have failed or pending series; resolve these before building the subset.')
    return selected, index, config

selected, source_index, preprocessing_config = inspect_source(NORMALIZED_ROOT, set(verified[ID]))
display(selected.groupby(ID).size().rename('normalized_series').to_frame())
print('Ready to copy:', len(selected), 'series across', selected[ID].nunique(), 'studies')


Inspected shard 000 - selected series: 5
Inspected shard 001 - selected series: 5
Inspected shard 002 - selected series: 10
Inspected shard 003 - selected series: 6
Inspected shard 004 - selected series: 0
Inspected shard 005 - selected series: 10
Inspected shard 006 - selected series: 11
Inspected shard 007 - selected series: 4
Inspected shard 008 - selected series: 0
Inspected shard 009 - selected series: 20
Inspected shard 010 - selected series: 25
Inspected shard 011 - selected series: 5
Inspected shard 012 - selected series: 0


,normalized_series
StudyInstanceUID,
1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866,4
1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097,4
1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636,5
1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471,5
1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773,5
1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801,5
1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585,5
1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062,5
1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576,5


Ready to copy: 101 series across 20 studies


## 3. Copy only the selected image caches
Each NPZ is copied byte-for-byte and checked for valid shape, masks, finite pixels, study/series IDs and preprocessing identity. Output is built in a temporary directory and renamed only after all checks pass. Existing output is never overwritten.

In [5]:
def build_subset(root, output, labels, series, source_index, prep, label_text):
    root, output = Path(root), Path(output)
    if output.exists():
        raise FileExistsError(f'{output} already exists. Use a new output folder to avoid mixing builds.')
    output.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='.radiologist-subset-', dir=output.parent) as temp:
        stage = Path(temp) / 'dataset'
        stage.mkdir()
        manifest = series.copy().reset_index(drop=True)
        copied_paths, hashes, byte_counts = [], [], []
        with ExitStack() as stack:
            archives = {name: stack.enter_context(tarfile.open(root / name, 'r:'))
                        for name in manifest.source_package.unique()}
            total = sum(safe_file(archives[r.source_package], r.cache_path).size for r in manifest.itertuples())
            if shutil.disk_usage(stage).free < total + MIN_FREE_GB * 1e9:
                raise OSError(f'Insufficient free disk for {total / 1e9:.2f} GB of caches plus reserve.')
            for number, row in enumerate(manifest.itertuples(), 1):
                arc = archives[row.source_package]
                member = safe_file(arc, row.cache_path)
                rel = Path('cache') / source_index['identity']['run_id'] / Path(row.cache_path).name
                target = stage / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                if target.exists():
                    raise ValueError('Duplicate cache filename in selected series.')
                digest = hashlib.sha256()
                with arc.extractfile(member) as src, target.open('wb') as dst:
                    for chunk in iter(lambda: src.read(8 << 20), b''):
                        dst.write(chunk); digest.update(chunk)
                if target.stat().st_size != member.size:
                    raise ValueError('Incomplete cache copy.')
                with np.load(target, allow_pickle=False) as z:
                    meta = json.loads(str(z['meta'].item()))
                    image = z['image']
                    if meta.get(ID) != getattr(row, ID) or meta.get(SID) != getattr(row, SID):
                        raise ValueError('Cache IDs do not match the series manifest.')
                    if meta.get('run_id') != source_index['identity']['run_id']:
                        raise ValueError('Cache preprocessing identity mismatch.')
                    if list(image.shape) != meta['shape'] or not np.isfinite(image).all():
                        raise ValueError('Invalid image shape or nonfinite pixels.')
                    if z['valid_bits'].size * 8 < image.size:
                        raise ValueError('Invalid packed pixel mask.')
                copied_paths.append(rel.as_posix()); hashes.append(digest.hexdigest()); byte_counts.append(member.size)
                print(f'Copied {number}/{len(manifest)} series', end='\r')
        manifest['source_cache_path'] = manifest['cache_path']
        manifest['cache_path'] = copied_paths
        manifest['sha256'] = hashes
        manifest['cache_bytes'] = byte_counts
        manifest.to_csv(stage / 'series_manifest.csv', index=False)
        labels.to_csv(stage / 'radiologist_labels.csv', index=False)
        training = labels.copy()
        for target in TARGETS:
            known = training[target].notna()
            training[target + '__mask'] = known.astype(int)
            training[target + '__gold'] = known.astype(int)
            training[target + '__weight'] = known.astype(float)
            training[target + '__source'] = np.where(known, 'radiologist', 'unknown')
        # Deliberately no fold column: decide the evaluation/training role explicitly.
        training.to_csv(stage / 'study_labels.csv', index=False)
        counts = manifest.groupby(ID).size().rename('series_count').reset_index()
        counts.to_csv(stage / 'selected_studies.csv', index=False)
        (stage / 'preprocessing_config.json').write_text(json.dumps(prep, indent=2))
        info = dict(schema_version=1, study_count=len(labels), series_count=len(manifest),
                    targets=TARGETS, source_dataset=SOURCE_HANDLE, source_identity=source_index['identity'],
                    source_index_sha256=hashlib.sha256((root/'dataset_index.json').read_bytes()).hexdigest(),
                    radiologist_csv_sha256=hashlib.sha256(label_text.encode('utf-8')).hexdigest(),
                    split='unassigned', normalized_images_copied_without_changes=True)
        (stage / 'subset_manifest.json').write_text(json.dumps(info, indent=2))
        (stage / 'README.md').write_text('# Radiologist-labelled normalized MRI subset\n\n'
            'Only the selected studies are included. Images are copied without renormalization. '
            'Labels come exclusively from the supplied radiologist CSV. Blank targets are unknown. '
            'Use series_manifest.csv for relative cache paths and study_labels.csv for targets, masks and weights. '
            'No training/validation split is assigned. This is a standalone subset, not a cumulative shard bundle. '
            'Keep private and comply with the source competition data terms.\n')
        stage.rename(output)
    return info

summary = build_subset(NORMALIZED_ROOT, OUTPUT_DIR, verified, selected,
                       source_index, preprocessing_config, RADIOLOGIST_LABELS_TEXT)
print('\nCreated:', OUTPUT_DIR)
print(json.dumps(summary, indent=2))


Copied 101/101 series
Created: /kaggle/working/rsna-knee-radiologist-20
{
  "schema_version": 1,
  "study_count": 20,
  "series_count": 101,
  "targets": [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
  ],
  "source_dataset": "gany24558/rsna-knee-normalized-all-training-data",
  "source_identity": {
    "run_id": "6029db4ffe8efb0de063",
    "plan_id": "86fa019e9e464c574370990dfb0269d4e1d40093fcfbf6676ccd10011a758886",
    "label_sha256": "7238636c87769beb21e451e490bf398bc45eff763db64dd7bdc5e5e981bec08f",
    "num_shards": 13
  },
  "source_index_sha256": "e9c83cbc54c339c465823c8404eb1b96fcfebc0fe221bebf37798ffd78eb43dd",
  "radiologist_csv_sha256": "8d9123a1bcb0a1e9a01372b81397aebfb7955e3d232492d897f4066eadf9c238",
  "split": "unassigned",
  "normalized_images_copied_without_changes": true
}


## 4. Read the result and save as a dataset

The output contains actual normalized pixels, not pointers to the original dataset. After verification, run Section 5 to publish the output folder as a Kaggle Dataset. Only this folder is needed by downstream users.

```text
rsna-knee-radiologist-20/
  cache/<run_id>/*.npz
  radiologist_labels.csv
  study_labels.csv
  series_manifest.csv
  selected_studies.csv
  preprocessing_config.json
  subset_manifest.json
  README.md
```

The previous `load_published_dataset` helper expects cumulative tar shards and is not the reader for this standalone folder. Load its tables and NPZs directly as shown below. For independent evaluation, ensure the selected studies (and other studies from the same patients, if known) were excluded from model training.


In [6]:
subset_series = pd.read_csv(OUTPUT_DIR / 'series_manifest.csv', dtype={ID: str, SID: str})
subset_labels = pd.read_csv(OUTPUT_DIR / 'study_labels.csv', dtype={ID: str})
assert subset_labels[ID].nunique() == EXPECTED_STUDIES
assert set(subset_series[ID]) == set(subset_labels[ID])
example_path = OUTPUT_DIR / subset_series.iloc[0].cache_path
with np.load(example_path, allow_pickle=False) as cache:
    example_image = cache['image']
    example_valid = np.unpackbits(cache['valid_bits'], count=example_image.size).reshape(example_image.shape).astype(bool)
    example_meta = json.loads(str(cache['meta'].item()))
print('Example series shape:', example_image.shape)
print('Verified subset:', len(subset_labels), 'studies;', len(subset_series), 'series')


Example series shape: (64, 320, 320)
Verified subset: 20 studies; 101 series


## 5. Publish the verified subset to Kaggle

Run this after Section 4 succeeds. This cell is self-contained: if the completed output folder still exists, run just this cell without rebuilding the caches. Enable Internet and use your authenticated Kaggle session. If `kagglehub` is missing, run `%pip install kagglehub` first.

It creates **gany24558/rsna-knee-radiologist-verified-20** (new datasets are private), or uploads a new version if that handle already exists. Keep the dataset private. Existing dataset visibility is not changed. All image caches and tables are uploaded; the source all-studies dataset is untouched. The local folder retains its earlier name so the completed run can be published immediately.

Wait for Kaggle processing after upload submission. If publishing fails, keep the output folder and retry only this cell. A retry may create another version if the preceding upload actually succeeded.

API: [KaggleHub dataset upload](https://github.com/Kaggle/kagglehub#upload-a-dataset).


In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import kagglehub

SUBSET_OUTPUT = Path('/kaggle/working/rsna-knee-radiologist-20')
PUBLISH_HANDLE = 'gany24558/rsna-knee-radiologist-verified-20'


def validate_subset_for_upload(folder):
    folder = Path(folder)
    required = ['README.md', 'subset_manifest.json', 'preprocessing_config.json',
                'radiologist_labels.csv', 'study_labels.csv', 'series_manifest.csv',
                'selected_studies.csv']
    for name in required:
        if not (folder / name).is_file():
            raise FileNotFoundError(f'Missing {name}. Complete the subset build or restore its saved output folder.')
    info = json.loads((folder / 'subset_manifest.json').read_text())
    labels = pd.read_csv(folder / 'radiologist_labels.csv', dtype={'StudyInstanceUID': str})
    manifest = pd.read_csv(folder / 'series_manifest.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    if info['study_count'] != 20 or len(labels) != 20 or not labels.StudyInstanceUID.is_unique:
        raise ValueError('Expected exactly 20 unique labelled studies.')
    if set(labels.StudyInstanceUID) != set(manifest.StudyInstanceUID):
        raise ValueError('Image/label study IDs differ.')
    if len(manifest) != info['series_count'] or manifest.duplicated(['StudyInstanceUID', 'SeriesInstanceUID']).any():
        raise ValueError('Incorrect or duplicated series inventory.')
    if not manifest.status.eq('ok').all():
        raise ValueError('Subset contains unsuccessful series.')
    expected_files = set(required)
    for row in manifest.itertuples():
        rel = Path(row.cache_path)
        if rel.is_absolute() or '..' in rel.parts or not rel.parts or rel.parts[0] != 'cache':
            raise ValueError('Invalid relative cache path.')
        path = folder / rel
        if path.is_symlink() or not path.is_file():
            raise ValueError(f'Missing or linked cache: {rel}')
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 << 20), b''):
                digest.update(chunk)
        if digest.hexdigest() != row.sha256 or path.stat().st_size != row.cache_bytes:
            raise ValueError(f'Cache checksum/size mismatch: {rel}')
        expected_files.add(rel.as_posix())
    actual_files = {p.relative_to(folder).as_posix() for p in folder.rglob('*') if p.is_file()}
    if actual_files != expected_files:
        raise ValueError('Output folder has missing or unexpected files; upload only the verified subset.')
    return info


publication_info = validate_subset_for_upload(SUBSET_OUTPUT)
print(f"Uploading {publication_info['study_count']} studies and {publication_info['series_count']} series to {PUBLISH_HANDLE}")
kagglehub.dataset_upload(
    PUBLISH_HANDLE,
    str(SUBSET_OUTPUT),
    version_notes=f"Radiologist-verified subset: 20 studies, {publication_info['series_count']} normalized series; embedded verified labels.",
)
print('Dataset version submitted: https://www.kaggle.com/datasets/' + PUBLISH_HANDLE)
print('Wait for Kaggle to finish processing before attaching the dataset to another notebook.')
